# Annotator agreement

## Inter-annotator agreement

### Read data

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("food101_offtopic.csv")
df

### Cohen's kappa

#### With `sklearn`

In [ ]:
import sklearn.metrics

In [ ]:
help(sklearn.metrics.cohen_kappa_score)

In [ ]:
choice_mapping = {"Not off-topic": 0, "Off-topic": 1}
annotator_mapping = {a: i for i, a in enumerate(sorted(df["annotator"].unique()))}

In [ ]:
# Apply the mapping
df["choice"] = df["choice"].map(choice_mapping)
df["annotator"] = df["annotator"].map(annotator_mapping)

In [ ]:
pivot_df = df.pivot(index="image", columns="annotator", values="choice")
pivot_df

In [ ]:
pivot_df_2 = pivot_df[[0, 2]]
pivot_df_2

In [ ]:
sklearn.metrics.cohen_kappa_score(pivot_df_2[0], pivot_df_2[2])

#### With `statmodels`

In [ ]:
import statsmodels.stats.inter_rater

In [ ]:
help(statsmodels.stats.inter_rater.cohens_kappa)

In [ ]:
confusion_matrix = sklearn.metrics.confusion_matrix(pivot_df[0], pivot_df[2])
confusion_matrix

In [ ]:
table_, bins_ = statsmodels.stats.inter_rater.to_table(pivot_df_2.values)
table_, bins_

In [ ]:
statsmodels.stats.inter_rater.cohens_kappa(table=confusion_matrix)

#### With `agreement`

In [ ]:
!pip install agreement

In [ ]:
import agreement.metrics

In [ ]:
help(agreement.metrics.cohens_kappa)

In [ ]:
df_2 = df[df["annotator"].isin((0, 2))]
df_2

In [ ]:
from agreement.utils.transform import pivot_table_frequency
answers_matrix = pivot_table_frequency(df_2["image"], df_2["choice"])
users_matrix = pivot_table_frequency(df_2["annotator"], df_2["choice"])

In [ ]:
agreement.metrics.cohens_kappa(answers_matrix=answers_matrix, users_matrix=users_matrix)

In [ ]:
# Uncontrolled application of Cohen's kappa with multiple annotators
answers_matrix = pivot_table_frequency(df["image"], df["choice"])
users_matrix = pivot_table_frequency(df["annotator"], df["choice"])
agreement.metrics.cohens_kappa(answers_matrix=answers_matrix, users_matrix=users_matrix)

### Scott's pi

#### With `agreement`

In [ ]:
help(agreement.metrics.scotts_pi)

In [ ]:
agreement.metrics.scotts_pi(answers_matrix=answers_matrix)

### Fleiss' kappa

#### With `statsmodels`

In [ ]:
help(statsmodels.stats.inter_rater.fleiss_kappa)

In [ ]:
table, categories = statsmodels.stats.inter_rater.aggregate_raters(data=pivot_df[[0, 2, 3]].values)

In [ ]:
statsmodels.stats.inter_rater.fleiss_kappa(table)

### Krippendorff's alpha

#### With `agreement`

In [ ]:
help(agreement.metrics.krippendorffs_alpha)

In [ ]:
agreement.metrics.krippendorffs_alpha(answers_matrix)

#### With `krippendorff`

In [ ]:
!pip install krippendorff

In [ ]:
import krippendorff
help(krippendorff.alpha)

In [ ]:
krippendorff.alpha(pivot_df.values.transpose())

## Intra-annotator agreement

### Read data

#### Brush mask

In [ ]:
from PIL import Image
import numpy as np
brush_image = Image.open("brush.png")
brush_array = np.array(brush_image)

In [ ]:
np.unique(brush_array, return_counts=True)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(brush_array)
plt.show()

#### Polygon mask

In [ ]:
!pip install pycocotools

In [ ]:
import json
with open("polygon.json", "r") as file:
    coco_data = json.load(file)
coco_data

In [ ]:
annotation = coco_data["annotations"][-1]

In [ ]:
segmentation = annotation["segmentation"]
images_data = coco_data["images"]
for image_data in images_data:
    if image_data["id"] == annotation["image_id"]:
        image_height = image_data["height"]
        image_width = image_data["width"]
        break
image_height, image_width

In [ ]:
import pycocotools.mask
rle = pycocotools.mask.frPyObjects(segmentation, image_height, image_width)
polygon_array = pycocotools.mask.decode(rle)
polygon_array.shape

In [ ]:
polygon_array = polygon_array.squeeze()
plt.imshow(polygon_array)
plt.show()

### Intersection over Union

In [ ]:
intersection = brush_array.astype(bool) & polygon_array.astype(bool)
union = brush_array.astype(bool) | polygon_array.astype(bool)
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(intersection)
axs[0].set_title('Intersection')
axs[1].imshow(union)
axs[1].set_title('Union')
plt.show()

In [ ]:
iou = intersection.sum() / union.sum()
iou